In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, confusion_matrix, classification_report
import numpy as np
from tqdm.auto import tqdm
import torch.nn.functional as F
from torchcam.methods import GradCAM
import random
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import sys

# Compatibility shim for old torchvision hub builds
if not hasattr(torch, '_six'):
    from unittest.mock import MagicMock
    mock = MagicMock()
    mock.string_classes = (str, bytes)
    torch._six = mock
    sys.modules['torch._six'] = mock

: 

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {device}')

In [ ]:
class TinyAgCNN(nn.Module) :
    def __init__(self) :
        super(TinyAgCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3,stride=1, padding=0) ,
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(16, 32, kernel_size=3,stride=1, padding=0) ,
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3,stride=1, padding=0) ,
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(6400,128),
            nn.ReLU(),
            nn.Linear(128,4)
        )
    def forward(self, x) :
        x = self.features(x)
        x = torch.flatten(x,1)
        x = self.classifier(x)
        return x

In [ ]:
import torch.nn.functional as F
def kd_loss(student_logits, teacher_logits, labels, alpha, T):
    HL = F.cross_entropy(student_logits, labels)
    psl, pt = (F.log_softmax(student_logits/T, dim=1), F.softmax(teacher_logits/T, dim=1))
    SL = F.kl_div(psl,pt, reduction="batchmean") * T**2
    loss = alpha * HL + (1-alpha) * SL
    return loss

In [ ]:
def train_distill(student:nn.Module, teacher:nn.Module, train_loader:DataLoader, optimizer, device, alpha:float, T:float, epochs:int):
    student.train()
    teacher.eval()
    
    for epoch in range(epochs):
        rl = 0.0
        all_preds, all_labels = [], []
        for inputs, labels in train_loader:
            inputs,labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            with torch.no_grad() :
                teacher_logts = teacher(inputs)
            student_logts = student(inputs)
            _, preds = torch.max(student_logts, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            loss = kd_loss(student_logts, teacher_logts, labels, alpha, T)
            loss.backward()
            optimizer.step()
            rl += loss.item()
            
        epoch_recall = recall_score(all_labels, all_preds, average='macro')
        print(f"Epoch {epoch+1}/{epochs} | Loss: {rl/len(train_loader):.4f} | Recall: {epoch_recall:.4f}")
    return rl / len(train_loader), all_preds, all_labels